# Model set-up


In [15]:
import torch
import librosa
import soundfile as sf
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

# Load the processor and model
MODEL_NAME = "mrrubino/wav2vec2-large-xlsr-53-l2-arctic-phoneme" # wav2vec based phoneme trascriber trained on L2-ARTIC
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_NAME)

# Check device availability
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [18]:
# Load and preprocess the audio file
def load_audio(audio_path, target_sr=16000):
  """Load an audio file and resample it to 16kHz."""
  audio, sr = librosa.load(audio_path, sr=target_sr)
  return audio

# Utils


In [2]:
# Original ARPAbet to IPA mapping from SoapBox Labs
arpabet_to_ipa = {
    "AA": "a", "AE": "æ", "AH": "ʌ", "AO": "ɔ", "AW": "aʊ", "AY": "aɪ",
    "EH": "ɛ", "ER": "ɚ", "EY": "eɪ", "IH": "ɪ", "IY": "i", "OW": "oʊ",
    "OY": "ɔɪ", "UH": "ʊ", "UW": "u", "B": "b", "CH": "t͡ʃ", "D": "d",
    "DH": "ð", "F": "f", "G": "ɡ", "HH": "h", "JH": "dʒ", "K": "k",
    "L": "l", "M": "m", "N": "n", "NG": "ŋ", "P": "p", "R": "ɹ",
    "S": "s", "SH": "ʃ", "T": "t", "TH": "θ", "V": "v", "W": "w",
    "Y": "j", "Z": "z", "ZH": "ʒ"
}

# Invert the dictionary to map IPA to ARPAbet
ipa_to_arpabet = {v: k for k, v in arpabet_to_ipa.items()}

def convert_ipa_to_arpabet(ipa_words):
    """
    Convert a list of IPA words (strings of concatenated phonemes) to ARPAbet words.

    :param ipa_words: List of IPA words where each word is a string of concatenated phonemes.
    :return: List of lists, where each inner list contains ARPAbet phonemes for a word.
    """
    arpabet_words = []
    for word in ipa_words:
        # Break the word into phonemes
        phonemes = []  # Collect matched phonemes
        i = 0
        while i < len(word):
            matched = False
            # Match multi-character IPA phonemes first
            for ipa_phoneme in sorted(ipa_to_arpabet.keys(), key=len, reverse=True):
                if word[i:].startswith(ipa_phoneme):
                    phonemes.append(ipa_to_arpabet[ipa_phoneme])
                    i += len(ipa_phoneme)
                    matched = True
                    break
            # If no match, add an unknown marker and move forward
            if not matched:
                phonemes.append("<UNK>")
                i += 1
        # Append the list of phonemes for the word
        arpabet_words.append(phonemes)
    return arpabet_words

In [3]:
import re

def remove_numbers_from_phonemes(phon_list):
    """
    Remove all numbers from phonemes in a nested list.

    Parameters:
        phon_list (list of lists): Nested list of phonemes.

    Returns:
        list of lists: Updated nested list with numbers removed from phonemes.
    """
    cleaned_phon_list = []
    for word_phonemes in phon_list:
        cleaned_word = [re.sub(r'\d', '', phoneme) for phoneme in word_phonemes]
        cleaned_phon_list.append(cleaned_word)
    return cleaned_phon_list

In [4]:
# alignment
import numpy as np

def align_phoneme_sequences(truth_words, uttered_words, gap_penalty=1, substitution_cost=1):
    """
    Align phoneme sequences separated by words.

    Parameters:
        truth_words (list of lists): Ground truth phoneme sequences grouped by words.
        uttered_words (list of lists): Uttered phoneme sequences grouped by words.
        gap_penalty (int): Penalty for gaps.
        substitution_cost (int): Cost for substitutions.

    Returns:
        alignment (list of tuples): Aligned phoneme sequences with '-' for gaps.
    """
    def align_two_sequences(seq1, seq2):
        """
        Align two sequences using dynamic programming.
        """
        n = len(seq1)
        m = len(seq2)
        dp = np.zeros((n + 1, m + 1))

        # Initialize DP table
        for i in range(n + 1):
            dp[i][0] = i * gap_penalty
        for j in range(m + 1):
            dp[0][j] = j * gap_penalty

        # Fill DP table
        for i in range(1, n + 1):
            for j in range(1, m + 1):
                match_cost = 0 if seq1[i - 1] == seq2[j - 1] else substitution_cost
                dp[i][j] = min(
                    dp[i - 1][j - 1] + match_cost,  # Match or substitution
                    dp[i - 1][j] + gap_penalty,    # Deletion
                    dp[i][j - 1] + gap_penalty     # Insertion
                )

        # Traceback to find alignment
        alignment_seq1 = []
        alignment_seq2 = []
        i, j = n, m
        while i > 0 or j > 0:
            if i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + (0 if seq1[i - 1] == seq2[j - 1] else substitution_cost):
                alignment_seq1.append(seq1[i - 1])
                alignment_seq2.append(seq2[j - 1])
                i -= 1
                j -= 1
            elif i > 0 and dp[i][j] == dp[i - 1][j] + gap_penalty:
                alignment_seq1.append(seq1[i - 1])
                alignment_seq2.append('-')
                i -= 1
            else:
                alignment_seq1.append('-')
                alignment_seq2.append(seq2[j - 1])
                j -= 1

        return alignment_seq1[::-1], alignment_seq2[::-1]

    # Align each word pair
    alignment = []
    for truth_word, uttered_word in zip(truth_words, uttered_words):
        aligned_truth, aligned_uttered = align_two_sequences(truth_word, uttered_word)
        alignment.append((aligned_truth, aligned_uttered))

    return alignment

In [5]:
def generate_phoneme_labels(data):
    """
    Generate phoneme labels for comparison of expected and uttered phonemes.

    Parameters:
    data (list of tuples): Each tuple contains (expected phonemes, uttered phonemes).

    Returns:
    list of tuples: Each tuple contains (phonemes, labels).
                    Phonemes are from the expected list, and labels are binary (0: correct, 1: incorrect).
    """
    results = []
    for expected, uttered in data:
        labels = [
            0 if exp == utt else 1
            for exp, utt in zip(expected, uttered)
        ]
        results.append((expected, labels))
    return results

In [6]:
from IPython.display import HTML, display

def display_phonemes_with_labels(phonemes, labels, word):
    """
    Display phonemes with their labels, along with the expected word.
    Incorrect phonemes (label=1) are displayed in red.

    Parameters:
    phonemes (list of str): List of phonemes to display.
    labels (list of int): Binary labels (0 for correct, 1 for incorrect).
    word (str): The expected word corresponding to the phonemes.
    """
    # Validate input
    if len(phonemes) != len(labels):
        raise ValueError("Phonemes and labels lists must be of the same length.")

    # Construct HTML for phonemes
    styled_phonemes = [
        f"<span style='color:red;'>{phoneme}</span>" if label == 1 else f"<span>{phoneme}</span>"
        for phoneme, label in zip(phonemes, labels)
    ]
    phoneme_content = " ".join(styled_phonemes)

    # Construct complete HTML
    html_content = f"<div style='font-size:20px;'>{phoneme_content} - <b>{word}</b></div>"

    # Display
    display(HTML(html_content))

In [7]:
import cmudict

def convert_words_to_phonemes(words, cmu_dict):
  phonemes = []
  for word in words:
    if word in cmu_dict:
      phonemes.extend(cmu_dict[word][0])  # Use the first phoneme representation
    else:
      phonemes.append('<UNK>')  # Append 'UNK' for unknown words
  return phonemes

# Run


In [19]:
import cmudict
cmu = cmudict.dict()

# Path to test audio file
audio_path = 'Audios/test5-bad.wav'  # Replace with your audio file path

# Define the script
transcript = "the person that sat on the floor is punched"

# Load audio and normalize
audio_input = load_audio(audio_path)
input_values = processor(audio_input, return_tensors="pt", sampling_rate=16000).input_values
input_values = input_values.to(device)

# Step 3: Perform inference
with torch.no_grad():
    logits = model(input_values).logits

# Step 4: Decode the phonemes
predicted_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

# Step 5: Print the transcription result
print("Predicted Phonemes:")
print(transcription)

Predicted Phonemes:
zʌ bʊsʌn zæt sæt ɔn zʌ flɔ i bʌn


In [20]:
pred_phons = convert_ipa_to_arpabet(transcription.split())
pred_phons

[['Z', 'AH'],
 ['B', 'UH', 'S', 'AH', 'N'],
 ['Z', 'AE', 'T'],
 ['S', 'AE', 'T'],
 ['AO', 'N'],
 ['Z', 'AH'],
 ['F', 'L', 'AO'],
 ['IY'],
 ['B', 'AH', 'N']]

In [21]:
trans_phons = [convert_words_to_phonemes([word], cmu) for word in transcript.split()]
trans_phons

[['DH', 'AH0'],
 ['P', 'ER1', 'S', 'AH0', 'N'],
 ['DH', 'AE1', 'T'],
 ['S', 'AE1', 'T'],
 ['AA1', 'N'],
 ['DH', 'AH0'],
 ['F', 'L', 'AO1', 'R'],
 ['IH1', 'Z'],
 ['P', 'AH1', 'N', 'CH', 'T']]

In [22]:
cleaned_trans_phons = remove_numbers_from_phonemes(trans_phons)

In [72]:
# Generate labels
alignment = align_phoneme_sequences(cleaned_trans_phons, pred_phons)
phoneme_labels = generate_phoneme_labels(alignment)

# Example: Display using the previous function
for idx, (phonemes, labels) in enumerate(phoneme_labels):
    display_phonemes_with_labels(phonemes, labels, transcript.split()[idx])

# Random Testings


In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import torch
import torchaudio
import numpy as np

audio_path = '/content/drive/MyDrive/Test Audio/test2.wav'
waveform, sample_rate = torchaudio.load(audio_path)

# Resample audio if necessary
if sample_rate != 16000:
    resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
    waveform = resampler(waveform)

# Preprocess the waveform
inputs = processor(waveform.squeeze().numpy(), sampling_rate=16000, return_tensors="pt", padding=True)

# Pass inputs through the model
with torch.no_grad():
  outputs = model(**inputs, output_hidden_states=True)

hidden_states = outputs.hidden_states
hidden_states_last = hidden_states[-1]  # Last layer's output hidden states
print(hidden_states_last.shape)  # Should be (batch_size, sequence_length, hidden_dim)

torch.Size([1, 160, 1024])


In [ ]:
outputs.hidden_states[0].shape

torch.Size([1, 160, 1024])